# 04_analyze_eleveld_outputs

This notebook performs the primary regression and figure-generation workflow for the Eleveld-based propofol exposure analysis.

## Purpose

The workflow loads the patient-level dataset containing Eleveld-based peak effect-site concentration estimates, applies the analytic cohort restrictions used in the manuscript, fits spline-based regression models for dose and exposure across age, generates adjusted predictions, and produces manuscript figures and tables.

## Main steps

1. Load the patient-level Eleveld simulation output dataset
2. Apply analytic exclusions and cohort restrictions used in the primary analysis
3. Define the age-adjusted Eleveld pharmacodynamic reference function
4. Fit spline-based regression models for:
   - weight-normalized induction dose
   - modeled peak effect-site concentration
   - age-associated overexposure probability
5. Generate standardized adjusted predictions using cohort-level mean covariates
6. Produce the main multidimensional age-exposure figure
7. Produce logistic overexposure probability figures
8. Generate manuscript-ready table outputs and illustrative model-based predictions

## Main outputs

- adjusted age-exposure regression objects
- main manuscript figures derived from the Eleveld workflow
- manuscript-ready table outputs
- illustrative age-specific predicted values used in the Results section

## Important notes

- This notebook represents the primary regression workflow based on Eleveld-derived peak effect-site concentration estimates.
- The input dataset is expected to contain one row per patient or case after pharmacokinetic simulation.
- Model-based predictions are standardized using cohort-level mean covariate values, consistent with the manuscript analyses.
- Scratch cells and local sync/push utilities should be removed from the public version of the notebook.

## Reproducibility note

This notebook documents the primary Eleveld-based statistical analysis workflow used to generate the main manuscript figures and tables. Institution-specific storage paths and local synchronization utilities are omitted from the public version.

## Configuration and imports

This section defines the input file used for the primary Eleveld analysis and loads the Python packages required for regression modeling, prediction, and figure generation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import bs
from statsmodels.stats.proportion import proportion_confint

INPUT_FILE = "df_analysis_eleveld.pkl"



## Load Eleveld simulation output

This section loads the patient-level dataset generated by the Eleveld simulation workflow.

In [ ]:
df_analysis = pd.read_pickle(INPUT_FILE)

print(f"Loaded {len(df_analysis):,} rows from {INPUT_FILE}.")

## Apply primary analytic cohort restrictions

This section applies the age-based cohort restrictions used in the main analysis and reports cohort attrition for reproducibility and manuscript flow reporting.

In [ ]:
# 1. PREPARATION (18-89 Only with Attrition Reporting)
# ---------------------------------------------------
df = df_analysis.copy()

# Ensure AGE is numeric and capture initial unique patient count
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
initial_patients = df['OR_CASE_ID'].nunique()

# Step A: Filter for 18+
df = df[df["AGE"] >= 18].copy()
under_18_removed = initial_patients - df['OR_CASE_ID'].nunique()
post_pediatric_count = df['OR_CASE_ID'].nunique()

# Step B: Filter for < 90 (The specific request)
df = df[df["AGE"] < 90].copy()
over_90_removed = post_pediatric_count - df['OR_CASE_ID'].nunique()

# Print the attrition results for your STROBE diagram/Methods section
print(f"--- Attrition Report ---")
print(f"Initial unique patients: {initial_patients:,}")
print(f"Removed pediatric cases (<18): {under_18_removed:,}")
print(f"Removed geriatric outliers (>=90): {over_90_removed:,}")
print(f"Final analytic cohort (18-89): {df['OR_CASE_ID'].nunique():,}")
print("-" * 25)

# Standardize Sex (0=Male, 1=Female)
if df["SEX"].dtype == "object":
    df["SEX"] = df["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

# Define Bins (Note: upper bound 90 ensures 85-89 group is captured)
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

AGE_LO, AGE_HI = float(df["AGE"].min()), float(df["AGE"].max())

## Fit adjusted exposure and dose models

This section defines the Eleveld age-adjusted pharmacodynamic reference function, prepares the regression dataset, creates prespecified age strata, and fits spline-based models for peak modeled effect-site concentration and weight-normalized induction dose.

In [ ]:
# Eleveld Target Formula
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

# 1) PREPARATION (18-89 Only)
# ----------------------------
df = df_analysis.copy()
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df = df[(df["AGE"] >= 18) & (df["AGE"] < 90)].copy()

if df["SEX"].dtype == "object":
    df["SEX"] = df["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

bins   = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

df_reg = df.dropna(subset=["Max_Effect_Concentration", "AGE", "ASA_SCORE", "BMI", "SEX"]).copy()

# 2) MODELING
# ------------
AGE_LO, AGE_HI = float(df_reg["AGE"].min()), float(df_reg["AGE"].max())
formula_ce = f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_ce = smf.ols(formula_ce, data=df_reg).fit()

g_asa, g_bmi, g_sex = df_reg[["ASA_SCORE", "BMI", "SEX"]].mean()

# Predictions
age_grid = np.linspace(18, 89, 400)
grid_df = pd.DataFrame({"AGE": age_grid, "ASA_SCORE": g_asa, "BMI": g_bmi, "SEX": g_sex})
pred_grid = model_ce.get_prediction(grid_df).summary_frame()

bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_ce = model_ce.get_prediction(bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)).summary_frame()

# 3) VISUALIZATION: EXPOSURE
# ----------------------------
plt.figure(figsize=(15, 8.5)) # Increased height slightly
plt.plot(age_grid, pred_grid["mean"], linewidth=4, color='#1f77b4', label="Adj. Predicted Peak $C_e$")
plt.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color='red', label="Eleveld $Ce_{50}$ Target")

plt.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=3.5, color='#5dade2', alpha=0.6, edgecolor='#1f77b4',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"], bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]], capsize=5)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    # Mean Value Label
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.12, f'{row["mean"]:.2f}', ha='center', fontsize=11, fontweight='bold', color='#1f77b4')
    # Surplus Percentage Label (Higher padding)
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.38, f'+{diff:.0f}%', ha='center', fontsize=11, fontweight='bold', color='red')

plt.xticks(bin_stats["mean_age"], labels)
plt.title("Standardized Peak Exposure ($C_e$) vs. Validated Target\n", fontsize=15, fontweight='bold', pad=40)
plt.ylabel("Effect-Site Concentration (mcg/mL)")
plt.ylim(0, 5.0) # Explicitly set headroom to prevent label clipping
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# 1) MODELING DOSE
# -----------------
formula_dose = f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_dose = smf.ols(formula_dose, data=df_reg).fit()

pred_grid_dose = model_dose.get_prediction(grid_df).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)).summary_frame()

# 2) VISUALIZATION: TITRATION (No Reference Lines)
# ----------------------------
plt.figure(figsize=(15, 8))
plt.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color='#2ca02c', label="Adj. Induction Dose")
plt.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=3.5, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c',
        yerr=[bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"], bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]], capsize=5)

for i, row in bin_pred_dose.iterrows():
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontsize=11, fontweight='bold', color='#2ca02c')

plt.xticks(bin_stats["mean_age"], labels)
plt.title("Standardized Clinician Induction Dose Titration (Ages 18-89)\n", fontsize=15, fontweight='bold', pad=20)
plt.ylabel("Induction Dose (mg/kg ABW)")
plt.ylim(0, max(bin_pred_dose["mean_ci_upper"]) + 0.5) # Dynamic headroom
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# 1. SETUP & STANDARDIZED MODELS
# ---------------------------------------------------------------------------
df = df_analysis.copy()
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df = df[(df["AGE"] >= 18) & (df["AGE"] < 90)].copy()
if df["SEX"].dtype == "object": df["SEX"] = df["SEX"].map({"M":0,"F":1,"m":0,"f":1})

bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

# Fit Models
AGE_LO, AGE_HI = float(df["AGE"].min()), float(df["AGE"].max())
model_ce = smf.ols(f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df).fit()
model_dose = smf.ols(f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df).fit()

# 2. CALCULATE NORMALIZED % DECAY
# ---------------------------------------------------------------------------
g_means = df[["ASA_SCORE", "BMI", "SEX"]].mean()
bin_stats = df.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
pred_df = bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_means[0], BMI=g_means[1], SEX=g_means[2])

# Predictions with CIs
ce_res = model_ce.get_prediction(pred_df).summary_frame()
dose_res = model_dose.get_prediction(pred_df).summary_frame()
target_ce50 = 3.08 * np.exp(-0.00635 * (bin_stats["mean_age"] - 35))

# Normalized to 18-24 group
b_ce, b_dose, b_target = ce_res['mean'].iloc[0], dose_res['mean'].iloc[0], target_ce50.iloc[0]

bin_stats["Dose_Pct"] = (dose_res['mean'] / b_dose) * 100
bin_stats["Ce_Pct"] = (ce_res['mean'] / b_ce) * 100
bin_stats["Ce_L"], bin_stats["Ce_U"] = (ce_res['mean_ci_lower'] / b_ce) * 100, (ce_res['mean_ci_upper'] / b_ce) * 100
bin_stats["Target_Pct"] = (target_ce50 / b_target) * 100

# 3. VISUALIZATION (PEER-REVIEW READY)
# ---------------------------------------------------------------------------
plt.figure(figsize=(14, 8))
x = np.arange(len(labels))

# Plot normalized lines
plt.plot(x, bin_stats["Dose_Pct"], marker='o', linewidth=4, color='#2ca02c', label="Clinician Titration (Induction $mg/kg$ % Baseline)")

# Blue line with Confidence Interval Band (Addressing Critique #3)
plt.plot(x, bin_stats["Ce_Pct"], marker='s', linewidth=4, color='#1f77b4', label="Modeled Peak Exposure (Predicted $C_e$ % Baseline)")
plt.fill_between(x, bin_stats["Ce_L"], bin_stats["Ce_U"], color='#1f77b4', alpha=0.15)

plt.plot(x, bin_stats["Target_Pct"], marker='D', linewidth=3, linestyle='--', color='red', label="Validated Age-Adjusted PD Reference ($Ce_{50}$ % Baseline)")

plt.xticks(x, labels)
plt.ylabel("Percentage of Young Adult (18-24) Baseline (%)", fontsize=12)
plt.title("Relative Divergence: Clinical Titration vs. Pharmacological Target\n(Standardized to Mean Covariates and 18-24 Baseline)", fontsize=14, fontweight='bold', pad=20)
plt.ylim(50, 115)
plt.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## Generate the main multidimensional age-exposure figure

This section generates the primary manuscript figure showing modeled peak effect-site concentration, weight-normalized induction dose, and age-adjusted pharmacodynamic requirement across the adult lifespan.

In [ ]:
# --- MASTER FIGURE 1: OPTIMIZED SPACING ---

# 1. Size 19 is the "Goldilocks" height—not too tall, not too smooshed
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 19))

# 2. Manual spacing control
# hspace=0.32 gives enough room for titles without wasting space
plt.subplots_adjust(hspace=0.32, top=0.95, bottom=0.08, left=0.1, right=0.95)

# --- PANEL A: EXPOSURE VS REQUIREMENT ---
ax1.plot(age_grid, pred_grid["mean"], linewidth=4, color='#1f77b4', label="Adj. Predicted Peak $C_e$")
ax1.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color='red', label="Eleveld $Ce_{50}$ Target")
ax1.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=3.5, color='#5dade2', alpha=0.6, edgecolor='#1f77b4',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"], bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]], capsize=5)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.12, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#1f77b4')
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.38, f'+{diff:.0f}%', ha='center', fontweight='bold', color='red')

ax1.set_title("A: Standardized Peak Exposure ($C_e$) vs. Validated Target", loc='left', fontsize=14, fontweight='bold')
ax1.set_ylabel("Effect-Site Concentration (mcg/mL)")
ax1.set_ylim(0, 5.5)
ax1.set_xticks(bin_stats["mean_age"])
ax1.set_xticklabels(labels)
ax1.legend(loc="upper right", frameon=True)

# --- PANEL B: TITRATION ---
ax2.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color='#2ca02c', label="Adj. Induction Dose")
ax2.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=3.5, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c',
        yerr=[bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"], bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]], capsize=5)

for i, row in bin_pred_dose.iterrows():
    ax2.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#2ca02c')

ax2.set_title("B: Standardized Clinician Induction Dose Titration", loc='left', fontsize=14, fontweight='bold')
ax2.set_ylabel("Induction Dose (mg/kg ABW)")
ax2.set_xticks(bin_stats["mean_age"])
ax2.set_xticklabels(labels)
ax2.legend(loc="upper right", frameon=True)

# --- PANEL C: RELATIVE DIVERGENCE ---
x_indices = np.arange(len(labels))
ax3.plot(x_indices, bin_stats["Dose_Pct"], marker='o', linewidth=4, color='#2ca02c', label="Clinician Titration (mg/kg % Baseline)")
ax3.plot(x_indices, bin_stats["Ce_Pct"], marker='s', linewidth=4, color='#1f77b4', label="Modeled Peak Exposure ($C_e$ % Baseline)")
ax3.fill_between(x_indices, bin_stats["Ce_L"], bin_stats["Ce_U"], color='#1f77b4', alpha=0.15)
ax3.plot(x_indices, bin_stats["Target_Pct"], marker='D', linewidth=3, linestyle='--', color='red', label="Validated PD Reference ($Ce_{50}$ % Baseline)")

ax3.set_title("C: The Titration-Response Discrepancy: Relative Divergence from Requirement", loc='left', fontsize=14, fontweight='bold')
ax3.set_ylabel("Percentage of Baseline (%)")
ax3.set_xlabel("Age Cohort (Years)", fontsize=13, labelpad=10) # X-label only on bottom
ax3.set_xticks(x_indices)
ax3.set_xticklabels(labels)
ax3.set_ylim(40, 115)
ax3.legend(loc='lower left', frameon=True)
ax3.grid(axis='y', linestyle='--', alpha=0.3)

# SAVE AS PDF
plt.savefig("figure_overexposure_analysis.pdf", bbox_inches='tight', dpi=300)
print("✅ Figure saved. Height=19, hspace=0.32.")
plt.show()

## Fit overexposure probability models

This section defines binary pharmacodynamic benchmark outcomes and fits logistic spline models to estimate the probability of exceeding age-specific and young-adult exposure thresholds across age.

In [ ]:



# 1. SETUP & BINARY EVENT DEFINITION
# ---------------------------------------------------------------------------
df_plot = df_analysis.copy()

# Ensure AGE is numeric
df_plot["AGE"] = pd.to_numeric(df_plot["AGE"], errors="coerce")

# FIX FOR SEX: If it's string-based (M/F), map it to numbers 0/1
# If it is already 0/1, this won't hurt.
if df_plot["SEX"].dtype == object:
    # This maps Male to 0 and Female to 1; adjust if your data uses different labels
    df_plot["SEX"] = df_plot["SEX"].map({'Male': 0, 'Female': 1, 'M': 0, 'F': 1})
else:
    df_plot["SEX"] = pd.to_numeric(df_plot["SEX"], errors="coerce")

# Ensure other covariates are numeric
df_plot["ASA_SCORE"] = pd.to_numeric(df_plot["ASA_SCORE"], errors="coerce")
df_plot["BMI"] = pd.to_numeric(df_plot["BMI"], errors="coerce")

# Drop rows with missing values and filter age
initial_count = len(df_plot)
df_plot = df_plot.dropna(subset=["AGE", "ASA_SCORE", "BMI", "SEX"]).copy()
df_plot = df_plot[(df_plot["AGE"] >= 18) & (df_plot["AGE"] < 90)].copy()
final_count = len(df_plot)

print(f"Rows before cleaning: {initial_count}")
print(f"Rows after cleaning: {final_count}")

if final_count == 0:
    raise ValueError("Error: df_plot is empty! Check if your SEX, ASA_SCORE, or BMI column names are correct or if they contain unparseable values.")

# Physiological target formula
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

# Calculate the constant Young Adult Ceiling (Baseline for 18-24)
ya_baseline_age = df_plot[df_plot["AGE"] < 25]["AGE"].mean()
YA_CEILING = ce50_eleveld(ya_baseline_age)

# Create the binary events
df_plot["Over_Ind"] = (df_plot["Max_Effect_Concentration"] > ce50_eleveld(df_plot["AGE"])).astype(int)
df_plot["Over_YA"] = (df_plot["Max_Effect_Concentration"] > YA_CEILING).astype(int)

# 2. CALCULATE BINNED STATS
# ---------------------------------------------------------------------------
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df_plot["Age_Group"] = pd.cut(df_plot["AGE"], bins=bins, labels=labels, right=False)

stats = df_plot.groupby("Age_Group", observed=True).agg(
    n_total=("AGE", "count"),
    n_ind=("Over_Ind", "sum"),
    n_ya=("Over_YA", "sum"),
    mean_age=("AGE", "mean")
).reset_index()

for key in ['ind', 'ya']:
    p = stats[f'n_{key}'] / stats['n_total']
    low, high = proportion_confint(stats[f'n_{key}'], stats['n_total'], method='wilson')
    stats[f'p_{key}'], stats[f'low_{key}'], stats[f'high_{key}'] = p, low*100, high*100

# 3. COVARIATE-ADJUSTED LOGISTIC MODELS
# ---------------------------------------------------------------------------
g_asa = df_plot["ASA_SCORE"].mean()
g_bmi = df_plot["BMI"].mean()
g_sex = df_plot["SEX"].mean()

# Note: Using C(SEX) if you want to treat it as a category,
# but since it's 0/1, direct use is fine for the 'mean' adjustment.
formula_ind = "Over_Ind ~ bs(AGE, df=5, lower_bound=18, upper_bound=89) + ASA_SCORE + BMI + SEX"
formula_ya = "Over_YA ~ bs(AGE, df=5, lower_bound=18, upper_bound=89) + ASA_SCORE + BMI + SEX"

model_ind = smf.logit(formula_ind, data=df_plot).fit(disp=0)
model_ya = smf.logit(formula_ya, data=df_plot).fit(disp=0)

# Create a prediction grid holding covariates at their means
age_grid = np.linspace(18, 89, 200)
grid_df = pd.DataFrame({
    "AGE": age_grid,
    "ASA_SCORE": g_asa,
    "BMI": g_bmi,
    "SEX": g_sex
})

line_p_ind = model_ind.predict(grid_df)
line_p_ya = model_ya.predict(grid_df)

# 4. FINAL COMPOSITE VISUALIZATION
# ---------------------------------------------------------------------------
plt.figure(figsize=(14, 9))

# A. Adjusted Modeled Lines
plt.plot(age_grid, line_p_ind * 100, color='red', linewidth=3,
         label="Adjusted Risk: Exceeding Individual Target ($C_e > Ce_{50}$)")

plt.plot(age_grid, line_p_ya * 100, color='#1f77b4', linewidth=3, linestyle='--',
         label=f"Adjusted Risk: Exceeding YA Ceiling ($C_e > {YA_CEILING:.2f}$)")

# B. Raw Observed Data Points
plt.errorbar(stats["mean_age"], stats["p_ind"] * 100,
             yerr=[stats["p_ind"]*100 - stats["low_ind"], stats["high_ind"] - stats["p_ind"]*100],
             fmt='o', color='red', alpha=0.4, markersize=8, capsize=4, label="Observed Binned Proportion")

plt.errorbar(stats["mean_age"], stats["p_ya"] * 100,
             yerr=[stats["p_ya"]*100 - stats["low_ya"], stats["high_ya"] - stats["p_ya"]*100],
             fmt='s', color='#1f77b4', alpha=0.4, markersize=8, capsize=4)

# C. Percentage Labels
for i, row in stats.iterrows():
    plt.text(row["mean_age"], row["high_ind"] + 2, f'{row["p_ind"]*100:.0f}%',
             ha='center', va='bottom', fontsize=9, fontweight='bold', color='red')
    plt.text(row["mean_age"], row["low_ya"] - 5, f'{row["p_ya"]*100:.0f}%',
             ha='center', va='top', fontsize=9, fontweight='bold', color='#1f77b4')

# D. Cosmetics
plt.axhline(y=50, color='gray', linestyle=':', alpha=0.5)


plt.xlim(18, 90)
plt.ylim(0, 115)
plt.xlabel("Patient Age (Years)", fontsize=13)
plt.ylabel("Probability of Over-exposure (%)", fontsize=13)
plt.title(f"Probability of Exceeding PD Targets (Covariate-Adjusted)\n"
          ,
          fontsize=15, fontweight='bold', pad=25)

plt.legend(loc='lower right', frameon=True, shadow=True, fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.2)
plt.tight_layout()

plt.savefig("figure_geriatric_failure_risk_adjusted.pdf", bbox_inches='tight', dpi=300)
plt.show()

## Generate manuscript tables and age-stratified summaries

This section produces age-stratified adjusted predictions, overshoot estimates, and manuscript-ready table outputs derived from the primary Eleveld analysis.

In [ ]:
# 1. Calculate population means for adjustment
g_asa = df_plot["ASA_SCORE"].mean()
g_bmi = df_plot["BMI"].mean()
g_sex = df_plot["SEX"].mean()

# 2. Create the prediction row for Age 75
target_age = 75
df_pred_75 = pd.DataFrame({
    "AGE": [target_age],
    "ASA_SCORE": [g_asa],
    "BMI": [g_bmi],
    "SEX": [g_sex]
})

# 3. Get prediction summaries (includes CIs)
# summary_frame() returns a 1-row DataFrame with all stats
res_ya_75 = model_ya.get_prediction(df_pred_75).summary_frame(alpha=0.05)
res_ind_75 = model_ind.get_prediction(df_pred_75).summary_frame(alpha=0.05)

# 4. Extract values by position (Safe from naming errors)
p_ya, l_ya, u_ya = res_ya_75.iloc[0, 0]*100, res_ya_75.iloc[0, 2]*100, res_ya_75.iloc[0, 3]*100
p_ind, l_ind, u_ind = res_ind_75.iloc[0, 0]*100, res_ind_75.iloc[0, 2]*100, res_ind_75.iloc[0, 3]*100

# 5. Output the results
print(f"--- Standardized Model-Based Risk at Age {target_age} ---")
print(f"Adjusted for Population Mean (ASA: {g_asa:.2f}, BMI: {g_bmi:.1f})")
print("-" * 50)
print(f"Exceeding Individual Ce50:   {p_ind:.1f}% (95% CI: {l_ind:.1f}% - {u_ind:.1f}%)")
print(f"Exceeding Young Adult Floor: {p_ya:.1f}% (95% CI: {l_ya:.1f}% - {u_ya:.1f}%)")

In [ ]:
# 1. Calculate the population means (same as used in your plotting code)
g_asa = df_plot["ASA_SCORE"].mean()
g_bmi = df_plot["BMI"].mean()
g_sex = df_plot["SEX"].mean()

# 2. Create the prediction row with ALL required columns
target_age = 75
df_pred_75 = pd.DataFrame({
    "AGE": [target_age],
    "ASA_SCORE": [g_asa],
    "BMI": [g_bmi],
    "SEX": [g_sex]
})

# 3. Get the predictions from your models
# This will now work because all covariates are present
prob_ya_75 = model_ya.predict(df_pred_75).iloc[0]
prob_ind_75 = model_ind.predict(df_pred_75).iloc[0]

# 4. Output the results
print(f"--- Standardized Model-Based Risk at Age {target_age} ---")
print(f"(Adjusted for Mean ASA: {g_asa:.2f}, BMI: {g_bmi:.1f}, and Sex)")
print(f"Probability of exceeding the Young Adult Ceiling: {prob_ya_75 * 100:.1f}%")
print(f"Probability of exceeding Individual Ce50: {prob_ind_75 * 100:.1f}%")

In [ ]:

# 1. SETUP STUDY PARAMETERS
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

# Assuming df_analysis is your loaded case-level dataframe
df = df_analysis.copy()
df['AGE'] = pd.to_numeric(df['AGE'], errors='coerce')
df = df[(df['AGE'] >= 18) & (df['AGE'] < 90)].copy()

if df['SEX'].dtype == 'object':
    df['SEX'] = df['SEX'].str.upper().map({'M': 0, 'F': 1})

# Define Standard Bins
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

# Clean for model
df_reg = df.dropna(subset=['Max_Effect_Concentration', 'ASA_SCORE', 'BMI', 'SEX']).copy()

# 2. STANDARDIZED EXPOSURE MODEL
g_asa, g_bmi, g_sex = df_reg[['ASA_SCORE', 'BMI', 'SEX']].mean()
model_ce = smf.ols("Max_Effect_Concentration ~ bs(AGE, df=5) + ASA_SCORE + BMI + SEX", data=df_reg).fit()

# Get binned predictions
bin_stats = df_reg.groupby("Age_Group", observed=True).agg(Mean_Age=("AGE", "mean")).reset_index()
pred_input = bin_stats.rename(columns={'Mean_Age': 'AGE'}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)
adj_ce = model_ce.get_prediction(pred_input).summary_frame()

# 3. CREATE EXPOSURE TABLE
table_exposure = pd.DataFrame({
    'Age_Group': bin_stats['Age_Group'],
    'N': df_reg.groupby("Age_Group", observed=True).size().values,
    'Peak_Ce_95CI': [f"{m:.2f} ({l:.2f}-{u:.2f})" for m, l, u in zip(adj_ce['mean'], adj_ce['mean_ci_lower'], adj_ce['mean_ci_upper'])],
    'Ce50_Target': ce50_eleveld(bin_stats['Mean_Age']).round(2),
    'Overshoot_Pct': (((adj_ce['mean'] / ce50_eleveld(bin_stats['Mean_Age'])) - 1) * 100).round(1)
})

table_exposure.to_csv("table_exposure_part.csv", index=False)
print("✅ Exposure part generated and saved as 'table_exposure_part.csv'")

In [ ]:
# 1. Calculate Standardized Predictions and CIs
adj_ce = model_ce.get_prediction(pred_input).summary_frame()
targets = ce50_eleveld(bin_stats['Mean_Age'])

# 2. Calculate Overshoot and its 95% CI
# Formula: ((Ce / Target) - 1) * 100
ov_mean = ((adj_ce['mean'] / targets) - 1) * 100
ov_low  = ((adj_ce['mean_ci_lower'] / targets) - 1) * 100
ov_high = ((adj_ce['mean_ci_upper'] / targets) - 1) * 100

# 3. Assemble Table with CIs
table_exposure = pd.DataFrame({
    'Age_Group': bin_stats['Age_Group'],
    'N': df_reg.groupby("Age_Group", observed=True).size().values,
    'Peak_Ce_95CI': [f"{m:.2f} ({l:.2f}-{u:.2f})" for m, l, u in zip(adj_ce['mean'], adj_ce['mean_ci_lower'], adj_ce['mean_ci_upper'])],
    'Ce50_Target': targets.round(2),
    'Overshoot_95CI': [f"{m:.1f}% ({l:.1f}-{u:.1f}%)" for m, l, u in zip(ov_mean, ov_low, ov_high)]
})

table_exposure.to_csv("table_exposure_part.csv", index=False)
print("✅ Exposure part with CIs generated and saved.")

In [ ]:

# 1. RE-DEFINE THE INPUTS (To prevent NameErrors)
# This ensures bin_pred_input exists in this specific cell's memory
bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_input = bin_stats.rename(columns={"mean_age": "AGE"}).assign(
    ASA_SCORE=df_reg["ASA_SCORE"].mean(),
    BMI=df_reg["BMI"].mean(),
    SEX=df_reg["SEX"].mean()
)

# 2. GENERATE PREDICTIONS
adj_ce = model_ce.get_prediction(bin_pred_input).summary_frame()
adj_dose = model_dose.get_prediction(bin_pred_input).summary_frame()

# Physiological requirement targets
targets = ce50_eleveld(bin_stats['mean_age'])
ov_mean = ((adj_ce['mean'] / targets) - 1) * 100
ov_low  = ((adj_ce['mean_ci_lower'] / targets) - 1) * 100
ov_high = ((adj_ce['mean_ci_upper'] / targets) - 1) * 100

# 3. BUILD THE DATAFRAME
table_exposure = pd.DataFrame({
    'Age_Group': labels,
    'N': [f"{int(x):,}" for x in df_reg.groupby("Age_Group", observed=True).size().values],
    'Dose': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(adj_dose['mean'], adj_dose['mean_ci_lower'], adj_dose['mean_ci_upper'])],
    'Peak_Ce': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(adj_ce['mean'], adj_ce['mean_ci_lower'], adj_ce['mean_ci_upper'])],
    'Target': [f"{t:.2f}" for t in targets],
    'Surplus': [f"{m:.1f}% ({l:.1f}--{u:.1f}%)" for m, l, u in zip(ov_mean, ov_low, ov_high)]
})

# 4. FORMAT HEADERS (Using Raw Strings to avoid SyntaxWarnings)
df_tex = table_exposure.copy()
df_tex.columns = [
    "Age Cohort",
    "N",
    "Induction Dose (mg/kg)",
    r"Peak $C_e$ (95\% CI)",
    r"Target $Ce_{50}$",
    r"Surplus (95\% CI)"
]

# 5. GENERATE & CLEAN LATEX
latex_table = df_tex.to_latex(index=False, escape=False, column_format='llcccc')
final_latex = latex_table.replace('%', r'\%').replace(r'\\%', r'\%')

# 6. SAVE
with open("table_exposure_data.tex", "w") as f:
    f.write(final_latex)

print("✅ table_exposure_data.tex updated successfully.")